# LangChain

**Module:** 09-llm-frameworks

**Notebook:** `02-langchain.ipynb`

This expanded lesson goes beyond definitions: each topic includes *why it matters*, *how it works*, intuition, pitfalls, and when to use it—plus runnable Python demos, comparison aids, and exercises.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain and apply **Core Concepts** with clear contracts and failure modes
- Explain and apply **LLMs & Chat Models** with clear contracts and failure modes
- Explain and apply **Prompt Components** with clear contracts and failure modes
- Explain and apply **Chains (Legacy Mental Model)** with clear contracts and failure modes
- Explain and apply **LCEL** with clear contracts and failure modes
- Explain and apply **Retrieval Components** with clear contracts and failure modes
- Explain and apply **Memory** with clear contracts and failure modes
- Explain and apply **Agents & Tools** with clear contracts and failure modes
- Evaluate tradeoffs (quality, cost, latency, safety) for designs in this lesson
- Implement small Python prototypes that make the ideas testable


## How to Use This Notebook

1. Read the topic sections fully—do not jump only to code.
2. Run each demo; then change inputs to break them and fix them.
3. API examples use placeholders like `YOUR_API_KEY` or `os.environ.get(...)`.
4. Keep secrets out of git; treat prompts/tool schemas as versioned code.
5. Complete the **Try It Yourself** exercises before moving on.


### Pipeline walkthrough — LangChain

```mermaid
flowchart LR
  A[Problem / user goal] --> B[Contract: IO + constraints]
  B --> C[Implement core path]
  C --> D[Validate / guardrails]
  D --> E[Eval fixtures]
  E --> F[Observe in production]
  F -->|regressions| B
```

```text
goal -> contract -> implement -> validate -> evaluate -> monitor -> revise
```


## Curriculum Map

This notebook's spine (preserve/cover all of these):

1. **Core Concepts**
2. **LLMs & Chat Models**
3. **Prompt Components**
4. **Chains (Legacy Mental Model)**
5. **LCEL**
6. **Retrieval Components**
7. **Memory**
8. **Agents & Tools**
9. **Monitoring**

Read top-to-bottom once, then revisit weak spots with the exercises.


## Core Concepts

### Definition
**Core Concepts** is a core building block in 02-langchain within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around Core Concepts typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Core Concepts: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain Core Concepts as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Core Concepts as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Core Concepts
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Core Concepts when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.

### Quick reference

| Lens | Question |
|------|----------|
| Product | What user outcome does Core Concepts improve? |
| Engineering | What is the interface / data contract? |
| Safety | What can go wrong if it fails open? |
| Ops | How will we notice regressions? |


In [ ]:
# Demo: make "Core Concepts" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Core Concepts"
    notebook: str = "02-langchain"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_0 = ConceptContract()
print(json.dumps({"contract": asdict(contract_0), "health": contract_0.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Core Concepts"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Core Concepts"}
strong = {"definition": "Core Concepts", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Core Concepts"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Core Concepts", "passed": len(checks)-len(failed), "failed": failed})


In [ ]:
# Demo: decision table for applying "Core Concepts"
options = [
    {"option": "baseline_simple", "quality": 0.7, "cost": 1, "ops": 0.9},
    {"option": "advanced_core_concept", "quality": 0.85, "cost": 3, "ops": 0.6},
]
for o in options:
    o["utility"] = round(o["quality"] * 2 - 0.3*o["cost"] + 0.5*o["ops"], 3)
best = max(options, key=lambda x: x["utility"])
print("ranked:", sorted(options, key=lambda x: -x["utility"]))
print("prefer:", best["option"])


## LLMs & Chat Models

### Definition
**LLMs & Chat Models** is a core building block in 02-langchain within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around LLMs & Chat Models typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For LLMs & Chat Models: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain LLMs & Chat Models as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating LLMs & Chat Models as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for LLMs & Chat Models
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use LLMs & Chat Models when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "LLMs & Chat Models" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "LLMs & Chat Models"
    notebook: str = "02-langchain"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_1 = ConceptContract()
print(json.dumps({"contract": asdict(contract_1), "health": contract_1.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "LLMs & Chat Models"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "LLMs & Chat Models"}
strong = {"definition": "LLMs & Chat Models", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "LLMs & Chat Models"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "LLMs & Chat Models", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — LLMs & Chat Models

**Situation:** A team wants to productionize a feature involving **LLMs & Chat Models**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Prompt Components

### Definition
**Prompt Components** is a core building block in 02-langchain within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around Prompt Components typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Prompt Components: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain Prompt Components as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Prompt Components as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Prompt Components
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Prompt Components when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Prompt Components" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Prompt Components"
    notebook: str = "02-langchain"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_2 = ConceptContract()
print(json.dumps({"contract": asdict(contract_2), "health": contract_2.health()}, indent=2))


In [ ]:
# MCP-like component registry (pedagogical)
registry = {
    "resources": [{"uri": "doc://readme", "name": "README", "mimeType": "text/plain"}],
    "tools": [{"name": "search", "inputSchema": {"type": "object", "properties": {"q": {"type": "string"}}}}],
    "prompts": [{"name": "explain", "arguments": [{"name": "topic"}]}],
}

def list_caps():
    return {k: [x.get("name") or x.get("uri") for x in v] for k, v in registry.items()}

print(list_caps())


In [ ]:
# JSON-RPC style message shapes used conceptually by MCP
msg_request = {"jsonrpc": "2.0", "id": 1, "method": "tools/call", "params": {"name": "search", "arguments": {"q": "SSO"}}}
msg_response = {"jsonrpc": "2.0", "id": 1, "result": {"content": [{"type": "text", "text": "SSO allowlist..."}]}}
msg_error = {"jsonrpc": "2.0", "id": 1, "error": {"code": -32601, "message": "Method not found"}}
print(msg_request["method"], "=>", msg_response["result"]["content"][0]["text"][:40])


## Chains (Legacy Mental Model)

### Definition
**Chains (Legacy Mental Model)** is a core building block in 02-langchain within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around Chains (Legacy Mental Model) typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Chains (Legacy Mental Model): (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain Chains (Legacy Mental Model) as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Chains (Legacy Mental Model) as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Chains (Legacy Mental Model)
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Chains (Legacy Mental Model) when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Chains (Legacy Mental Model)" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Chains (Legacy Mental Model)"
    notebook: str = "02-langchain"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_3 = ConceptContract()
print(json.dumps({"contract": asdict(contract_3), "health": contract_3.health()}, indent=2))


In [ ]:
class Pipeline:
    def __init__(self):
        self.stages = []
    def add(self, name, fn):
        self.stages.append((name, fn)); return self
    def run(self, data):
        audit = []
        for name, fn in self.stages:
            data = fn(data)
            audit.append({"stage": name, "type": type(data).__name__})
        return data, audit

result, audit = (
    Pipeline()
    .add("normalize", lambda s: s.strip().lower())
    .add("tokens", lambda s: s.split())
    .add("features", lambda toks: {"n": len(toks), "head": toks[:3]})
    .run("  SSO Login Loop  ")
)
print(result); print(audit)


In [ ]:
# Parallel fan-out / fan-in sketch
from concurrent.futures import ThreadPoolExecutor

def analyze(kind, text):
    return {"kind": kind, "len": len(text)}

text = "investigate checkout latency"
with ThreadPoolExecutor(max_workers=3) as ex:
    parts = list(ex.map(lambda k: analyze(k, text), ["security", "perf", "ux"]))
merged = {p["kind"]: p["len"] for p in parts}
print(merged)


### Worked scenario — Chains (Legacy Mental Model)

**Situation:** A team wants to productionize a feature involving **Chains (Legacy Mental Model)**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## LCEL

### Definition
**LCEL** is a core building block in 02-langchain within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around LCEL typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For LCEL: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain LCEL as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating LCEL as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for LCEL
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use LCEL when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "LCEL" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "LCEL"
    notebook: str = "02-langchain"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_4 = ConceptContract()
print(json.dumps({"contract": asdict(contract_4), "health": contract_4.health()}, indent=2))


In [ ]:
class Pipeline:
    def __init__(self):
        self.stages = []
    def add(self, name, fn):
        self.stages.append((name, fn)); return self
    def run(self, data):
        audit = []
        for name, fn in self.stages:
            data = fn(data)
            audit.append({"stage": name, "type": type(data).__name__})
        return data, audit

result, audit = (
    Pipeline()
    .add("normalize", lambda s: s.strip().lower())
    .add("tokens", lambda s: s.split())
    .add("features", lambda toks: {"n": len(toks), "head": toks[:3]})
    .run("  SSO Login Loop  ")
)
print(result); print(audit)


In [ ]:
# Parallel fan-out / fan-in sketch
from concurrent.futures import ThreadPoolExecutor

def analyze(kind, text):
    return {"kind": kind, "len": len(text)}

text = "investigate checkout latency"
with ThreadPoolExecutor(max_workers=3) as ex:
    parts = list(ex.map(lambda k: analyze(k, text), ["security", "perf", "ux"]))
merged = {p["kind"]: p["len"] for p in parts}
print(merged)


## Retrieval Components

### Definition
**Retrieval Components** is a core building block in 02-langchain within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around Retrieval Components typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Retrieval Components: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain Retrieval Components as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Retrieval Components as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Retrieval Components
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Retrieval Components when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Retrieval Components" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Retrieval Components"
    notebook: str = "02-langchain"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_5 = ConceptContract()
print(json.dumps({"contract": asdict(contract_5), "health": contract_5.health()}, indent=2))


In [ ]:
GOLDEN = [{"x": "charged twice", "y": "billing"}, {"x": "SSO", "y": "auth"}]

def predict(x: str) -> str:
    return "billing" if "charge" in x.lower() or "invoice" in x.lower() else ("auth" if "sso" in x.lower() or "login" in x.lower() else "other")

def eval_prompt(name: str):
    rows = [(g["x"], g["y"], predict(g["x"])) for g in GOLDEN]
    acc = sum(y == p for _, y, p in rows) / len(rows)
    return {"name": name, "accuracy": acc, "rows": rows}

print(eval_prompt("v1"))


In [ ]:
import hashlib

def prompt_version(text: str) -> str:
    return "pv_" + hashlib.sha1(text.encode()).hexdigest()[:10]

a = "Label tickets carefully"
b = "Label tickets as billing|auth|outage|other. JSON only."
print(prompt_version(a), prompt_version(b))


### Worked scenario — Retrieval Components

**Situation:** A team wants to productionize a feature involving **Retrieval Components**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Memory

### Definition
**Memory** is a core building block in 02-langchain within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around Memory typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Memory: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain Memory as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Memory as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Memory
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Memory when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Memory" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Memory"
    notebook: str = "02-langchain"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_6 = ConceptContract()
print(json.dumps({"contract": asdict(contract_6), "health": contract_6.health()}, indent=2))


In [ ]:
from collections import deque

class MemoryStore:
    def __init__(self, k=4):
        self.short = deque(maxlen=k)
        self.summary = ""
        self.episodic = []

    def add_turn(self, role, content):
        self.short.append({"role": role, "content": content})
        if role == "user":
            self.episodic.append(content[:160])

    def pack(self):
        return {"summary": self.summary, "recent": list(self.short), "episodes": self.episodic[-5:]}

mem = MemoryStore()
mem.add_turn("user", "My plan is Pro")
mem.add_turn("assistant", "Noted: plan=Pro")
mem.summary = "User on Pro plan"
print(mem.pack())


In [ ]:
# Semantic memory stub: bag-of-words retrieval
DOCS = ["refund policy under $5", "SSO allowlist redirects", "P0 outage page oncall"]

def retrieve(q: str, k=2):
    qw = set(q.lower().split())
    scored = sorted(DOCS, key=lambda d: len(qw & set(d.split())), reverse=True)
    return scored[:k]

print(retrieve("need refund for small charge"))


## Agents & Tools

### Definition
**Agents & Tools** is a core building block in 02-langchain within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around Agents & Tools typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Agents & Tools: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain Agents & Tools as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Agents & Tools as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Agents & Tools
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Agents & Tools when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Agents & Tools" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Agents & Tools"
    notebook: str = "02-langchain"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_7 = ConceptContract()
print(json.dumps({"contract": asdict(contract_7), "health": contract_7.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


### Worked scenario — Agents & Tools

**Situation:** A team wants to productionize a feature involving **Agents & Tools**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Monitoring

### Definition
**Monitoring** is a core building block in 02-langchain within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around Monitoring typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Monitoring: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain Monitoring as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Monitoring as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Monitoring
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Monitoring when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Monitoring" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Monitoring"
    notebook: str = "02-langchain"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_8 = ConceptContract()
print(json.dumps({"contract": asdict(contract_8), "health": contract_8.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Monitoring"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Monitoring"}
strong = {"definition": "Monitoring", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Monitoring"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Monitoring", "passed": len(checks)-len(failed), "failed": failed})


## Comparison Snapshot

Use this table when reviewing designs in **LangChain**.

| Topic | Do | Don't |
|-------|----|-------|
| Core Concepts | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| LLMs & Chat Models | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Prompt Components | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Chains (Legacy Mental Model) | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| LCEL | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Retrieval Components | Design carefully; measure; bound cost | Skipping eval / unbounded loops |


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| Core Concepts | Key concept covered in this notebook; see its section for definition and pitfalls |
| LLMs & Chat Models | Key concept covered in this notebook; see its section for definition and pitfalls |
| Prompt Components | Key concept covered in this notebook; see its section for definition and pitfalls |
| Chains (Legacy Mental Model) | Key concept covered in this notebook; see its section for definition and pitfalls |
| LCEL | Key concept covered in this notebook; see its section for definition and pitfalls |
| Retrieval Components | Key concept covered in this notebook; see its section for definition and pitfalls |
| Memory | Key concept covered in this notebook; see its section for definition and pitfalls |
| Agents & Tools | Key concept covered in this notebook; see its section for definition and pitfalls |


## Summary & Key Takeaways

- **LangChain** is a production concern: contracts, evals, and guardrails beat vibe-driven prompting.
- Every major topic above includes definition, motivation, mechanism, intuition, pitfalls, and usage guidance—use that checklist in design reviews.
- Prefer small, measurable demos before framework sprawl.
- Bound loops, validate tool args, and keep API keys in environment variables (`YOUR_API_KEY` is a placeholder only).
- Carry forward: connect these ideas to the next notebooks in **09-llm-frameworks**.


## Try It Yourself

1. Implement a failing test/fixture for **Core Concepts**, then fix your demo until it passes.
2. Implement a failing test/fixture for **LLMs & Chat Models**, then fix your demo until it passes.
3. Implement a failing test/fixture for **Prompt Components**, then fix your demo until it passes.
4. Implement a failing test/fixture for **Chains (Legacy Mental Model)**, then fix your demo until it passes.
5. Implement a failing test/fixture for **LCEL**, then fix your demo until it passes.
6. Estimate token cost for your prompt/tool trace at 1k and 100k requests/day.
7. Write a 5-row comparison of two design alternatives from this notebook; pick one with explicit criteria.
8. Red-team your solution with empty input, hostile input, and a tool/API timeout.
